# Getting oriented before I write a real query

*Standalone -- no prior vignette required.*

I'm building toward an actual grant proposal, so unlike just poking around with a keyword, I want to know what I'm working with before I commit to anything. I don't want to guess at a column name and get it wrong, and I don't want to filter on a value that turns out to be spelled differently than I assumed. Before I touch `get_subject_data`, I want to look around.

In [ ]:
from cdapython import *
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt
opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0
opt.columnDefs = [{"className": "dt-left", "targets": "_all"}]

First, the biggest-picture view -- what tables even exist to search:

In [ ]:
tables()

I'm interested in age at diagnosis, but I don't know the exact column name offhand. Rather than guess, I can search by what a column is described as doing:

In [ ]:
columns(description='age')

`age_at_observation` looks right, and its description tells me it's recorded in years -- I didn't have to find that out the hard way by filtering and getting confused later. Before I filter on it, I want to see what the values actually look like -- is it populated often, are there extreme outliers, anything odd:

In [ ]:
column_values('age_at_observation', data_source='GDC')

That gives me a real sense of the range before I write a single filter. Now I want a count, not a list of rows -- how many subjects actually have both an adenocarcinoma diagnosis and a non-null age:

In [ ]:
summarize_subjects(match_all=['diagnosis = *adenocarcinoma*', 'age_at_observation != NULL'])

1,308 subjects. `!= NULL` here means "has a value at all," not an unusual edge case -- it's a normal, filterable condition. The reverse (`age_at_observation = NULL`) would show me everyone *missing* an age, which is its own useful check: if I required age on every search, how much data would I silently be throwing away?

Some of what matched here turned out to be mice, so I'll narrow further:

In [ ]:
summarize_subjects(match_all=['diagnosis = *adenocarcinoma*', 'age_at_observation != NULL', 'species = human'])

1,226 human subjects -- 82 of my original 1,308 turned out to be mice, filtered out with one added condition. That's a real, usable number. Now I want to look at age brackets specifically -- numeric columns support ordinary comparison operators, and I can chain two to express a range:

In [ ]:
summarize_subjects(match_all=['60 < age_at_observation <= 70', 'diagnosis = *adenocarcinoma*', 'species = human'])

493 subjects in that age bracket alone. I checked the table landscape, found the right column by description instead of guessing, looked at real values before filtering, and used `NULL` and numeric ranges deliberately rather than stumbling into them. None of this required knowing the underlying data model or any source's internal vocabulary -- just a willingness to look before leaping, which is the whole point of browsing first when the stakes are higher than idle curiosity.